# PolyWin R2 — v16: CROSS-TARGET DECODER (physics + learned sibling arms; fold-safe)

## Experiment (pre-registered)
* v14 (P14) froze the long-train baseline at **LB 0.883** after full-PI1M
  pretraining failed to add signal. v16 does NOT retrain anything.
* **v16 adds a decoder stage AFTER the frozen blend inputs:** per molecule, the
  trained OOF/test predictions for the seven targets (eea,egb,egc,ei,eps,nc,tg)
  are joined into one row keyed by canonical SMILES (`tr_piv`). Two supplementary
  prediction arms are built ONLY from those sibling values (physics + learned):

      physics :  egc = ei - eea            (thermodynamic identity, no fit)
                 egb = a * egc + b         (fitted on train pairs)
                 eps = a * nc^2 + b        (fitted on train pairs)
      learned : per-target Ridge(alpha=10) over the other 6 targets' sibling
                values, GroupKFold(canon)-fold-safe (a polymer's own labels
                never enter its own Ridge).

* Both arms are fold-safe (coefficients fitted on other-fold train pairs;
  GroupKFold on canon, GLOBAL_FOLDS = 5). No test labels are ever used.
* The base arms [GBM, MT-GNN] OOF/test are BIT-IDENTICAL to the frozen v14 run
  (CORE_A/CORE_B verbatim). Only the final per-target Ridge is widened to 4 arms
  `[GBM, MT-GNN, physics, learned]`, alpha re-tuned on OOF.
* Production inputs reused: observations, descriptors, fingerprints, twins,
  GNN seeds (42,999,2025), folds = 5, full-PI1M pretraining (P14 run).

## Decoder detail
* `decoder_v16.py` is embedded VERBATIM as one cell. `build_pivot_df`, `sibling_feature`,
  `_fit_linear`, `physics_arm`, `learned_arm` are the unit-tested source of truth
  (tests/test_decoder_v16.py).
* Rows whose canon has no sibling value for an arm stay NaN; the blend falls
  back to the **target mean** for those arms (inert placeholder vs the frozen
  v14 baseline — that arm then just re-weights the base arms).

## Success / fail (judged only after the Kaggle run)
* **PASS gate:** small-target OOF gain >= +0.003 vs v14 on the multi-labeled
  subset (rows with >=2 targets observed) AND no single target regresses more
  than -0.003.
* **Stretch:** LB 0.898+.
* **FAIL:** no gain or a regression -> freeze P14 (0.883), do NOT submit v16.


In [ ]:
import os, sys, time, gc, random, warnings
import subprocess, importlib.util

def ensure_pkg(pkg, import_name=None):
    name = import_name or pkg
    if importlib.util.find_spec(name) is None:
        print("installing", pkg, flush=True)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "--disable-pip-version-check", pkg])

for _p, _n in [("rdkit", "rdkit"), ("torch_geometric", "torch_geometric"),
               ("lightgbm", "lightgbm"), ("catboost", "catboost"), ("xgboost", "xgboost")]:
    ensure_pkg(_p, _n)

# --- CUDA probe / repair identical to v14 (P100 sm_60) ---
_probe = ('import torch;' + 'a=torch.zeros(4,device="cuda");b=a+1;torch.cuda.synchronize();print("OK")')
def _force_cuda():
    try:
        _r = subprocess.run([sys.executable, "-c", _probe], capture_output=True,
                            text=True, timeout=600)
    except Exception:
        _r = None
    if _r is not None and _r.returncode == 0 and "OK" in (_r.stdout or ""):
        return
    print("CUDA kernel missing; installing torch 2.5.1 (cu121, supports P100 sm_60)...", flush=True)
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "--no-cache-dir", "--index-url",
                               "https://download.pytorch.org/whl/cu121", "torch==2.5.1"],
                              timeout=1800)
        _r2 = subprocess.run([sys.executable, "-c", _probe], capture_output=True,
                             text=True, timeout=600)
        print("post-reinstall probe rc:", _r2.returncode, flush=True)
    except Exception as _e:
        print("torch reinstall errored:", repr(_e)[:200], flush=True)
if os.path.exists("/kaggle"):
    _force_cuda()

import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GINEConv, global_mean_pool, global_add_pool
from rdkit import Chem
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")
from rdkit.Chem import Descriptors, AllChem, MACCSkeys, rdMolDescriptors, Crippen, GraphDescriptors
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

SMOKE = os.environ.get("SMOKE", "0") == "1"
SEED = 42
GNN_SEEDS = os.environ.get("GNN_SEEDS", "42,999,2025")
os.environ["GNN_SEEDS"] = GNN_SEEDS
print("GNN_SEEDS =", GNN_SEEDS, flush=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
def _cuda_ok():
    if not torch.cuda.is_available():
        return False
    try:
        a = torch.zeros(4, device="cuda"); b = a + 1; torch.cuda.synchronize(); del a, b
        return True
    except Exception:
        return False
DEVICE = "cuda" if _cuda_ok() else "cpu"
print("device:", DEVICE, flush=True)
GLOBAL_FOLDS = 5
MAX_EPOCHS = 120
PATIENCE = 20
EARLY_HOLDOUT = 0.15
BS = 256
LR = 1e-3
if DEVICE == "cpu":
    GLOBAL_FOLDS = min(GLOBAL_FOLDS, 2)
    MAX_EPOCHS = min(MAX_EPOCHS, 12)
    BS = 256
    PATIENCE = min(PATIENCE, 6)
PRETRAIN_EPOCHS = 10
PRETRAIN_SAMPLE = 2000000

if os.path.exists("/kaggle"):
    WORK = "/kaggle/working"; INP = "/kaggle/input"
else:
    WORK = os.path.join("vault", "pipeline_out_v16")
    INP = "official_dataset"
os.makedirs(WORK, exist_ok=True)
PRETRAINED = os.path.join(WORK, "pretrained_encoder.pt")
OUT = WORK

print("----- v16 CONFIG -----", flush=True)
print("PRETRAIN_SAMPLE =", PRETRAIN_SAMPLE, "| PRETRAIN_EPOCHS =", PRETRAIN_EPOCHS, flush=True)
print("device:", DEVICE, "| SMOKE:", SMOKE, "| folds:", GLOBAL_FOLDS,
      "| out:", OUT, flush=True)


In [ ]:
from sklearn.linear_model import Ridge

TARGETS = ["eea", "egb", "egc", "ei", "eps", "nc", "tg"]
TARGET_IDX = {t: i for i, t in enumerate(TARGETS)}


## 1. Data — current-round CSVs, canonicalize, compute descriptors + fingerprints

In [ ]:
def find_input(base, name):
    for p in [os.path.join(base, name), os.path.join(base, "ppp-round-2", name),
              os.path.join(base, "competitions", "ppp-round-2", name)]:
        if os.path.exists(p):
            return p
    return None

def canonical(s):
    if not isinstance(s, str):
        return None, None, None
    m = Chem.MolFromSmiles(s)
    if m is None:
        return None, None, None
    try:
        c = Chem.MolToSmiles(m)
        ik = Chem.MolToInchiKey(m)
    except Exception:
        return Chem.MolToSmiles(m), None, None
    return c, ik, m

def canon_fast(s):
    """MolToSmiles only (identical string to canonical(s)[0]) for the PI1M
    pretrain corpus — avoids 995k MolToInchiKey computations, no result change."""
    if not isinstance(s, str):
        return None
    try:
        m = Chem.MolFromSmiles(s)
        return Chem.MolToSmiles(m) if m is not None else None
    except Exception:
        return None

def feats(m):
    if m is None:
        return [np.nan] * 22
    return [
        Descriptors.MolWt(m), Descriptors.MolLogP(m), Descriptors.TPSA(m),
        Descriptors.NumHDonors(m), Descriptors.NumHAcceptors(m),
        Descriptors.RingCount(m), Descriptors.NumAromaticRings(m),
        Descriptors.NumAliphaticRings(m), Descriptors.NumSaturatedRings(m),
        Descriptors.NumRotatableBonds(m), rdMolDescriptors.CalcNumHeavyAtoms(m),
        Descriptors.NumHeteroatoms(m), Descriptors.FractionCSP3(m),
        Crippen.MolMR(m), rdMolDescriptors.CalcNumBridgeheadAtoms(m),
        rdMolDescriptors.CalcNumSpiroAtoms(m),
        rdMolDescriptors.CalcNumAromaticAtoms(m) if hasattr(rdMolDescriptors, "CalcNumAromaticAtoms") else Descriptors.NumAromaticRings(m),
        GraphDescriptors.BalabanJ(m), GraphDescriptors.Ipc(m),
        rdMolDescriptors.CalcNumLipinskiHBA(m), rdMolDescriptors.CalcNumLipinskiHBD(m),
        rdMolDescriptors.CalcNumAtomStereoCenters(m),
    ]

FNAMES = ["MolWt", "LogP", "TPSA", "HDon", "HAccep", "RingCnt", "AroRing", "AliRing", "SatRing",
          "RotB", "HeavyAt", "HeteroAt", "FracCSP3", "MR", "Bridge", "Spiro", "AroAt",
          "BalabanJ", "Ipc", "LipHBA", "LIHBD", "Stereo"]
assert len(FNAMES) == 22

train_path = find_input(INP, "train.csv")
test_path = find_input(INP, "test.csv")
assert train_path and test_path, "train.csv / test.csv not found in " + INP

tr = pd.read_csv(train_path)
te = pd.read_csv(test_path)
print("train:", tr.shape, "test:", te.shape, flush=True)

tcpl = tr["smiles"].map(canonical)
tr["canon"], tr["inchikey"], _ = zip(*tcpl)
tepl = te["smiles"].map(canonical)
te["canon"], te["inchikey"], _ = zip(*tepl)

tr_f = np.array(tr["smiles"].map(lambda s: feats(Chem.MolFromSmiles(s))).tolist())
te_f = np.array(te["smiles"].map(lambda s: feats(Chem.MolFromSmiles(s))).tolist())
tr[FNAMES] = tr_f
te[FNAMES] = te_f
print("descriptors done", flush=True)

trf = tr.dropna(subset=["target"]).copy()
tef = te.copy()

FEAT_COLS = [c for c in trf.columns if c not in
            ("smiles", "target", "target_type", "canon", "inchikey", "id")]
print("FEAT_COLS:", len(FEAT_COLS), flush=True)


In [ ]:
def add_fingerprints(df):
    morgan = np.zeros((len(df), 2048), dtype=np.float32)
    maccs = np.zeros((len(df), 167), dtype=np.float32)
    for i, s in enumerate(df["smiles"]):
        m = Chem.MolFromSmiles(s)
        if m is None:
            continue
        morgan[i] = np.frombuffer(AllChem.GetMorganFingerprintAsBitVect(
            m, 2, nBits=2048).ToBitString().encode(), "u1") - ord("0")
        maccs[i] = np.frombuffer(MACCSkeys.GenMACCSKeys(m).ToBitString().encode(),
                                 "u1") - ord("0")
    return morgan, maccs

F32_MAX = np.finfo(np.float32).max

def clean_feats(df):
    D = np.clip(df[FEAT_COLS].values, -F32_MAX, F32_MAX)
    for j in range(D.shape[1]):
        col = D[:, j]
        med = np.median(col[np.isfinite(col)]) if np.isfinite(col).any() else 0.0
        col[~np.isfinite(col)] = med
    return D.astype(np.float32)

D_tr = clean_feats(trf)
mor_tr, mc_tr = add_fingerprints(trf)
X = np.hstack([D_tr, mor_tr, mc_tr]).astype(np.float32)
Xs = StandardScaler().fit(X).transform(X).astype(np.float32)

D_te = clean_feats(tef)
mor_te, mc_te = add_fingerprints(tef)
Xte = np.hstack([D_te, mor_te, mc_te]).astype(np.float32)
Xtes = StandardScaler().fit(X).transform(Xte).astype(np.float32)

Y = trf["target"].values.astype(np.float32)
T = trf["target_type"].values
G = trf["canon"].values.astype(str)

idx_of_target = {t: np.where(T == t)[0] for t in TARGETS}
print("train:", X.shape, "test:", Xte.shape, "targets:", TARGETS, flush=True)


## 2. Level-0 sources (verbatim from mt_gnn_v2.py: graph feats + GINE + MT-GNN)

In [ ]:
# Graph featurization (MUST match the v10 pretrain kernel so the saved
# pretrained_encoder.pt loads into the same GINEEncoder).
# =====================================================================
ATOM_SYMBOLS = ["C", "N", "O", "S", "F", "Cl", "Br", "I", "Si", "P", "OTHER"]
HYBRIDIZATIONS = ["SP", "SP2", "SP3", "SP3D", "SP3D2", "OTHER"]
BOND_TYPES = ["SINGLE", "DOUBLE", "TRIPLE", "AROMATIC"]


def one_hot(value, choices):
    vec = [0.0] * len(choices)
    idx = choices.index(value) if value in choices else len(choices) - 1
    vec[idx] = 1.0
    return vec


def atom_features(atom):
    return (one_hot(atom.GetSymbol(), ATOM_SYMBOLS)
            + one_hot(atom.GetHybridization().name, HYBRIDIZATIONS)
            + [atom.GetIsAromatic() * 1.0, atom.IsInRing() * 1.0,
               atom.GetDegree() / 4.0, atom.GetTotalNumHs() / 4.0,
               atom.GetFormalCharge() / 2.0])


N_ATOM_FEATS = len(ATOM_SYMBOLS) + len(HYBRIDIZATIONS) + 5
N_BOND_FEATS = len(BOND_TYPES) + 2


def bond_features(bond):
    return one_hot(bond.GetBondType().name, BOND_TYPES) + [
        bond.GetIsConjugated() * 1.0, bond.IsInRing() * 1.0]


def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or mol.GetNumAtoms() < 2:
        return None
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    edge_index, edge_attr = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bf = bond_features(bond)
        edge_index += [[i, j], [j, i]]
        edge_attr += [bf, bf]
    if len(edge_index) == 0:
        edge_index = [[0, 0]]; edge_attr = [[0.0] * N_BOND_FEATS]
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_attr, dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr)


def build_graphs(df, has_target=True):
    out = {}
    freq = df["target_type"].value_counts(normalize=True)
    for row_id, row in zip(df.index, df.itertuples()):
        g = smiles_to_graph(row.smiles)
        if g is None:
            continue
        g.row_id = row_id
        g.smiles = row.smiles
        if has_target:
            g.target_idx = torch.tensor([TARGET_IDX[row.target_type]], dtype=torch.long)
            g.y = torch.tensor([float(row.target)], dtype=torch.float)
            g.w = torch.tensor([1.0 / freq[row.target_type]], dtype=torch.float)
        out[row_id] = g
    return out


def to_pyg(graphs):
    if isinstance(graphs, dict):
        graphs = list(graphs.values())
    return Batch.from_data_list(graphs)


t0 = time.time()
train_graphs = build_graphs(trf, has_target=True)
test_graphs = build_graphs(tef, has_target=False)
print(f"graphs: {len(train_graphs)} train, {len(test_graphs)} test "
      f"({time.time()-t0:.0f}s)", flush=True)


# =====================================================================
# Shared encoder + multi-task trunk (same GINEEncoder as v10 kernel).
# =====================================================================
class GINEEncoder(nn.Module):
    def __init__(self, n_atom_feats, n_bond_feats, hidden=128, n_layers=4, dropout=0.2):
        super().__init__()
        self.atom_encoder = nn.Linear(n_atom_feats, hidden)
        self.bond_encoder = nn.ModuleList(
            [nn.Linear(n_bond_feats, hidden) for _ in range(n_layers)])
        self.convs = nn.ModuleList(); self.bns = nn.ModuleList()
        for _ in range(n_layers):
            mlp = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                nn.Linear(hidden, hidden))
            self.convs.append(GINEConv(mlp, edge_dim=hidden))
            self.bns.append(nn.BatchNorm1d(hidden))
        self.dropout = dropout

    def forward(self, x, edge_index, edge_attr):
        h = self.atom_encoder(x)
        for conv, bn, bond_enc in zip(self.convs, self.bns, self.bond_encoder):
            e = bond_enc(edge_attr)
            h = conv(h, edge_index, e)
            h = bn(h); h = F.relu(h); h = F.dropout(h, p=self.dropout,
                                                    training=self.training)
        return h


class MTGNN(nn.Module):
    """Shared trunk + per-target heads. Optional cross-target twin features
    are concatenated to the pooled embedding before the shared trunk."""

    def __init__(self, n_atom_feats, n_bond_feats, n_twin=0, hidden=128,
                 n_layers=4, dropout=0.2):
        super().__init__()
        self.encoder = GINEEncoder(n_atom_feats, n_bond_feats, hidden,
                                   n_layers, dropout)
        pool_in = hidden * 2 + n_twin
        self.trunk = nn.Sequential(
            nn.Linear(pool_in, hidden), nn.BatchNorm1d(hidden), nn.ReLU(),
            nn.Dropout(dropout))
        self.heads = nn.ModuleList([
            nn.Sequential(nn.Linear(hidden, 64), nn.ReLU(), nn.Dropout(dropout),
                          nn.Linear(64, 1))
            for _ in TARGETS])

    def forward(self, data, twin=None):
        h = self.encoder(data.x, data.edge_index, data.edge_attr)
        pooled = torch.cat([global_mean_pool(h, data.batch),
                            global_add_pool(h, data.batch)], dim=1)
        if twin is not None and twin.size(1) > 0:
            pooled = torch.cat([pooled, twin.to(pooled.device)], dim=1)
        ht = self.trunk(pooled)
        out = torch.empty(data.batch.max() + 1, len(TARGETS), device=h.device)
        for i, head in enumerate(self.heads):
            out[:, i] = head(ht)[:, 0]
        return out

    def load_encoder(self, state_dict):
        enc = {k[len("encoder."):]: v for k, v in state_dict.items()
               if k.startswith("encoder.")}
        missing, unexpected = self.encoder.load_state_dict(enc, strict=False)
        print(f"  encoder init: missing={len(missing)} unexpected={len(unexpected)}",
              flush=True)


# =====================================================================

## 3. Pretrain the GINE encoder on FULL PI1M (P14 — NOT retrained here; same weights)

In [ ]:
pl_path = find_input(INP, "PI1M.csv")
pl = []
if pl_path:
    pldf = pd.read_csv(pl_path)
    smi_col = "SMILES" if "SMILES" in pldf.columns else "smiles"
    pldf = pldf[[smi_col]].rename(columns={smi_col: "smiles"})
    t0pl = time.time()
    pldf["canon"] = pldf["smiles"].map(canon_fast)
    print(f"PI-1M canonicalized in {time.time()-t0pl:.0f}s "
          f"({len(pldf)} rows, parsed {pldf['canon'].notna().sum()})", flush=True)
    pldf = pldf.dropna(subset=["canon"])
    pl = pldf.drop_duplicates("canon")["smiles"].tolist()
    print("PI-1M unique canons:", len(pl), flush=True)
    rng = np.random.RandomState(SEED); rng.shuffle(pl)
    pl = pl[:PRETRAIN_SAMPLE]
    print("PI-1M full corpus:", len(pl), "SMILES gen", flush=True)
else:
    print("no PI-1M: pretraining skipped", flush=True)

def build_pretrain_graphs_chunked(smiles_list, chunk=50000):
    graphs = []
    t0 = time.time()
    for c0 in range(0, len(smiles_list), chunk):
        chunk_g = []
        for smi in smiles_list[c0:c0+chunk]:
            g = smiles_to_graph(smi)
            if g is not None:
                chunk_g.append(g)
        graphs.extend(chunk_g)
        del chunk_g
        gc.collect()
        print(f"  graphs {len(graphs)}/{len(smiles_list)} "
              f"({time.time()-t0:.0f}s)", flush=True)
    return graphs

pl_graphs = build_pretrain_graphs_chunked(pl) if pl else []
print("pretraining graphs (full PI-1M):", len(pl_graphs), flush=True)

from torch_geometric.loader import DataLoader

class PretrainedEncoder(nn.Module):
    # identical to the v14 baseline wrapper; keys start 'encoder.'
    def __init__(self, n_atom_feats, n_bond_feats, hidden=128, n_layers=4,
                 mask_atom=0.15, mask_bond=0.20):
        super().__init__()
        self.encoder = GINEEncoder(n_atom_feats, n_bond_feats, hidden, n_layers)
        self.atom_proj = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                       nn.Linear(hidden, n_atom_feats))
        self.bond_proj = nn.Sequential(nn.Linear(2 * hidden, hidden), nn.ReLU(),
                                       nn.Linear(hidden, n_bond_feats))
        self.mask_atom = mask_atom; self.mask_bond = mask_bond

    def forward(self, x, edge_index, edge_attr, batch):
        n = x.size(0); m = edge_index.size(1)
        atom_mask = torch.rand(n, device=x.device) < self.mask_atom
        bond_mask = torch.rand(m, device=x.device) < self.mask_bond
        x_c = x.clone(); x_c[atom_mask] = 0.0
        ea_c = edge_attr.clone(); ea_c[bond_mask] = 0.0
        h = self.encoder(x_c, edge_index, ea_c)
        if atom_mask.any():
            atom_loss = F.mse_loss(self.atom_proj(h[atom_mask]), x[atom_mask])
        else:
            atom_loss = torch.zeros((), device=x.device)
        if bond_mask.any():
            src = h[edge_index[0, bond_mask]]; dst = h[edge_index[1, bond_mask]]
            if src.numel() > 0:
                bond_loss = F.mse_loss(self.bond_proj(torch.cat([src, dst], dim=1)), edge_attr[bond_mask])
            else:
                bond_loss = torch.zeros((), device=x.device)
        else:
            bond_loss = torch.zeros((), device=x.device)
        return atom_loss, bond_loss

def pretrain(epochs=PRETRAIN_EPOCHS, batch_size=1024, lr=1e-3):
    if len(pl_graphs) == 0:
        print("No PI-1M graphs - pretraining skipped", flush=True)
        return None
    model = PretrainedEncoder(N_ATOM_FEATS, N_BOND_FEATS).to(DEVICE)
    loader = DataLoader(pl_graphs, batch_size=batch_size, shuffle=True,
                        pin_memory=(DEVICE == "cuda"))
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    best = np.inf; best_state = None; t0 = time.time()
    for epoch in range(epochs):
        model.train(); tot_a = 0.0; tot_b = 0.0; nbl = 0
        for batch in loader:
            batch = batch.to(DEVICE)
            opt.zero_grad()
            a_loss, b_loss = model(batch.x, batch.edge_index, batch.edge_attr, batch)
            loss = a_loss + 0.5 * b_loss
            loss.backward(); opt.step()
            tot_a += a_loss.item(); tot_b += b_loss.item(); nbl += 1
            del batch
        va = (tot_a + 0.5 * tot_b) / max(nbl, 1)
        if va < best:
            best = va; best_state = {k: v.clone() for k, v in model.state_dict().items()}
        print(f"pretrain ep {epoch+1}/{epochs}: loss={va:.4f} ({time.time()-t0:.0f}s)", flush=True)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    if best_state:
        torch.save(best_state, PRETRAINED)
        print("saved pretrained_encoder.pt (full PI-1M)", flush=True)
    return best_state

pretrained_state = pretrain()


## 4. Level-0 predictions (verbatim: leak-safe twins + MT-GNN fold OOF + GBM trio stack)

In [ ]:
# Twin source: per-target LGBM OOF (leak-safe) + fold-bagged test preds.
# twin_u(row i) = target-u LGBM's prediction on row i's features.
# =====================================================================
print("\n=== Twin source: per-target LGBM OOF (leak-safe) ===", flush=True)
lgb_test_te = np.zeros((len(Xte), len(TARGETS)), dtype=np.float32)
TARGET_MEAN = {t: float(Y[idx_of_target[t]].mean()) for t in TARGETS}


# For train, twin_u(row i) uses lgb_oof_all from the target-u LGBM. But
# lgb_oof_all is stored by row index only for target-u rows. For a row of
# target t, its target-u twin value is the OOF prediction of the model_u on
# THAT row's features - we approximate with the per-target model_u evaluated
# on every train row (OOF where available, fold-safe holdout elsewhere).
# Simplest leak-safe approach: evaluate each target-u LGBM on ALL train rows
# via a dedicated OOF-style pass below.
print("\n=== Building leak-safe twin feature matrices ===", flush=True)
twin_train = np.zeros((len(X), (len(TARGETS) - 1) * 2), dtype=np.float32)
twin_test = np.zeros((len(Xte), (len(TARGETS) - 1) * 2), dtype=np.float32)
col_map = {}
for u in TARGETS:
    col = 0
    for t2 in TARGETS:
        if t2 == u:
            continue
        col_map[(u, t2)] = (col, col + 1)
        col += 2
def leak_safe_oof_scores():
    """For each target u, score every train row with a model trained on a
    canon-group that excludes that row (grouped OOF across all targets)."""
    scores = np.full((len(X), len(TARGETS)), np.nan, dtype=np.float32)
    # canon -> group id per row, using one global fold assignment
    gkf = GroupKFold(n_splits=GLOBAL_FOLDS)
    row_fold = np.zeros(len(X), dtype=int)
    for f, (_, va) in enumerate(gkf.split(Xs, Y, G)):
        row_fold[va] = f
    for u in TARGETS:
        for f in range(GLOBAL_FOLDS):
            in_fold = np.where(row_fold == f)[0]
            out_fold = np.setdiff1d(np.arange(len(X)), in_fold)
            idx_u_out = np.intersect1d(out_fold, idx_of_target[u])
            if len(idx_u_out) == 0:
                continue
            fit_ids, ho_ids = train_test_split(idx_u_out,
                                               test_size=EARLY_HOLDOUT,
                                               random_state=SEED)
            m = lgb.LGBMRegressor(n_estimators=800, learning_rate=0.05,
                                  num_leaves=15, min_child_samples=10,
                                  subsample=0.8, colsample_bytree=0.8,
                                  random_state=SEED, verbose=-1)
            m.fit(Xs[fit_ids], Y[fit_ids], eval_set=[(Xs[ho_ids], Y[ho_ids])])
            scores[in_fold, TARGET_IDX[u]] = m.predict(Xs[in_fold])
            # test bag
            lgb_test_te[:, TARGET_IDX[u]] += m.predict(Xtes) / GLOBAL_FOLDS
    return scores, lgb_test_te


twin_scores, lgb_test_te = leak_safe_oof_scores()
for t in TARGETS:
    for u in TARGETS:
        if u == t:
            continue
        iu = TARGET_IDX[u]
        c0, c1 = col_map[(t, u)]
        impute = TARGET_MEAN[u]
        v = twin_scores[:, iu]
        miss = np.isnan(v).astype(np.float32)
        v = np.where(miss, impute, v)
        twin_train[:, c0] = v; twin_train[:, c1] = miss
        # test: fold-bagged model_u prediction, always available
        tv = lgb_test_te[:, iu]
        tmiss = np.isnan(tv).astype(np.float32)
        tv = np.where(tmiss, impute, tv)
        twin_test[:, c0] = tv; twin_test[:, c1] = tmiss
print("twin matrices:", twin_train.shape, twin_test.shape, flush=True)


# =====================================================================
# MT-GNN fold-safe OOF + test bag
# =====================================================================
def early_split(fit_ids):
    uniq_g = np.unique(G[fit_ids])
    uniq_f, uniq_h = train_test_split(uniq_g, test_size=EARLY_HOLDOUT,
                                      random_state=SEED)
    return (fit_ids[np.isin(G[fit_ids], uniq_f)],
            fit_ids[np.isin(G[fit_ids], uniq_h)])


row_to_graph = {g.row_id: g for g in train_graphs.values()}
print("\n=== MT-GNN v2 (pretrained-init trunk + twins) ===", flush=True)
pretrained_state = torch.load(PRETRAINED, map_location="cpu") if os.path.exists(
    PRETRAINED) else None
if pretrained_state is not None:
    print("loaded pretrained_encoder.pt", flush=True)

GNN_SEEDS = [int(s) for s in os.environ.get("GNN_SEEDS", "42").split(",") if s.strip()]


def run_gnn_seed(seed):
    """One seed's MT-GNN: fold-safe GroupKFold OOF + fold-bagged test preds.
    Returns (mt_oof_all, mt_test) in raw scale. Identical math to the v13 run
    except torch/np/random seeding are reset per seed."""
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    n_twin = twin_train.shape[1]
    mt_oof_all = np.full(len(X), np.nan, dtype=np.float32)
    mt_test_folds = np.zeros((len(Xte), GLOBAL_FOLDS), dtype=np.float32)
    for f, (tr_idx, va_idx) in enumerate(GroupKFold(n_splits=GLOBAL_FOLDS).split(
            Xs, Y, G)):
        t0f = time.time()
        stats = {}
        y_norm = np.empty(len(tr_idx), dtype=np.float32)
        for t in TARGETS:
            mask = (T[tr_idx] == t)
            if mask.sum() > 0:
                mu, sd = Y[tr_idx][mask].mean(), Y[tr_idx][mask].std() + 1e-6
                stats[t] = (mu, sd)
                y_norm[mask] = (Y[tr_idx][mask] - mu) / sd
        fit_ids, ho_ids = early_split(tr_idx)
        model = MTGNN(N_ATOM_FEATS, N_BOND_FEATS, n_twin=n_twin).to(DEVICE)
        if pretrained_state is not None:
            model.load_encoder(pretrained_state)
        opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
        pos_of = {int(o): p for p, o in enumerate(tr_idx)}
        pos_of_all = {int(o): p for p, o in enumerate(tr_idx)}

        def predict_ids(ids, m=model):
            m.eval()
            out = np.empty(len(ids), dtype=np.float32)
            with torch.no_grad():
                for i in range(0, len(ids), 256):
                    bi = ids[i:i + 256]
                    graphs = [row_to_graph[int(b)] for b in bi]
                    batch = to_pyg(graphs).to(DEVICE)
                    twin = torch.tensor(twin_train[bi], dtype=torch.float)
                    p = m(batch, twin=twin).cpu().numpy()
                    for j, b in enumerate(bi):
                        ti = TARGET_IDX[T[b]]
                        mu, sd = stats[T[b]]
                        out[i + j] = p[j, ti] * sd + mu
            return out

        best, best_r2, pat = None, -np.inf, 0
        for ep in range(MAX_EPOCHS):
            model.train()
            perm = np.random.permutation(len(fit_ids))
            for i in range(0, len(perm), BS):
                bi = fit_ids[perm[i:i + BS]]
                idxs = [pos_of_all[int(b)] for b in bi]
                yb = torch.tensor(y_norm[idxs]).unsqueeze(1).to(DEVICE)
                wb = torch.tensor([row_to_graph[int(b)].w.item() for b in bi],
                                  dtype=torch.float).unsqueeze(1).to(DEVICE)
                graphs = [row_to_graph[int(b)] for b in bi]
                batch = to_pyg(graphs).to(DEVICE)
                twin = torch.tensor(twin_train[bi], dtype=torch.float)
                opt.zero_grad()
                pred = model(batch, twin=twin)
                ti = torch.tensor([TARGET_IDX[T[b]] for b in bi], device=DEVICE)
                pred_sel = pred.gather(1, ti.unsqueeze(1))
                loss = (F.mse_loss(pred_sel, yb, reduction="none") * wb).mean()
                loss.backward(); opt.step()
            hp = predict_ids(ho_ids)
            hr = r2_score(Y[ho_ids], hp)
            if hr > best_r2:
                best_r2 = hr
                best = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                pat = 0
            else:
                pat += 1
                if pat >= PATIENCE:
                    break
        model.load_state_dict(best)
        mt_oof_all[va_idx] = predict_ids(va_idx)
        # test prediction via graphs
        model.eval()
        with torch.no_grad():
            te_pred = np.zeros(len(Xte), dtype=np.float32)
            for i in range(0, len(Xte), 256):
                bi = np.arange(i, min(i + 256, len(Xte)))
                graphs = [test_graphs[int(b)] for b in bi]
                batch = to_pyg(graphs).to(DEVICE)
                twin = torch.tensor(twin_test[bi], dtype=torch.float)
                p = model(batch, twin=twin).cpu().numpy()
                for j, b in enumerate(bi):
                    ttt = tef["target_type"].iloc[int(b)]
                    ti = TARGET_IDX[ttt]
                    mu, sd = stats[ttt]
                    te_pred[i + j] = p[j, ti] * sd + mu
        mt_test_folds[:, f] = te_pred
        print(f"seed {seed}  fold {f}: holdout R2={best_r2:.4f} ({time.time()-t0f:.0f}s)", flush=True)
        del model; gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    assert not np.isnan(mt_oof_all).any()
    return mt_oof_all, mt_test_folds.mean(axis=1)


print("GNN_SEEDS =", GNN_SEEDS, flush=True)
mt_oof_sum = np.zeros(len(X), dtype=np.float32)
mt_test_sum = np.zeros(len(Xte), dtype=np.float32)
for _gs in GNN_SEEDS:
    _oo, _mt = run_gnn_seed(_gs)
    mt_oof_sum += _oo
    mt_test_sum += _mt
mt_oof_all = mt_oof_sum / len(GNN_SEEDS)
mt_test = mt_test_sum / len(GNN_SEEDS)
assert not np.isnan(mt_oof_all).any()

mt_oof = {t: mt_oof_all[idx_of_target[t]] for t in TARGETS}


# =====================================================================
# Per-target fallback vs the GBM trio stack (Ridge on lgb+xgb+cb).
# =====================================================================
print("\n=== GBM trio stack OOF (fallback floor) ===", flush=True)
gbm_oof = {t: {m: np.zeros(len(idx_of_target[t])) for m in ('lgb', 'xgb', 'cb')}
           for t in TARGETS}
gbm_test = {t: {m: np.zeros(len(Xte)) for m in ('lgb', 'xgb', 'cb')} for t in TARGETS}
import xgboost as xgb
import catboost as cb

for t in TARGETS:
    idx = idx_of_target[t]
    Xt, yt, gt = Xs[idx], Y[idx], G[idx]
    for tr_idx, va_idx in GroupKFold(n_splits=GLOBAL_FOLDS).split(Xt, yt, gt):
        fit_ids, ho_ids = early_split(tr_idx)
        l = lgb.LGBMRegressor(n_estimators=2000, learning_rate=0.03,
                              num_leaves=15, min_child_samples=10, subsample=0.8,
                              colsample_bytree=0.8, random_state=SEED, verbose=-1)
        x = xgb.XGBRegressor(n_estimators=2000, learning_rate=0.03, max_depth=4,
                             subsample=0.8, colsample_bytree=0.8, tree_method='hist',
                             random_state=SEED, verbosity=0)
        c = cb.CatBoostRegressor(iterations=2000, learning_rate=0.03, depth=6,
                                 random_seed=SEED, task_type='CPU', verbose=False,
                                 allow_writing_files=False)
        for m, est in ((l, l), (x, x), (c, c)):
            est.fit(Xt[fit_ids], yt[fit_ids], eval_set=[(Xt[ho_ids], yt[ho_ids])])
        gbm_oof[t]['lgb'][va_idx] = l.predict(Xt[va_idx])
        gbm_oof[t]['xgb'][va_idx] = x.predict(Xt[va_idx])
        gbm_oof[t]['cb'][va_idx] = c.predict(Xt[va_idx])
        gbm_test[t]['lgb'] += l.predict(Xtes) / GLOBAL_FOLDS
        gbm_test[t]['xgb'] += x.predict(Xtes) / GLOBAL_FOLDS
        gbm_test[t]['cb'] += c.predict(Xtes) / GLOBAL_FOLDS
    print(f"  {t} done", flush=True)

from sklearn.linear_model import Ridge

stack_oof = {}
stack_test = {}
for t in TARGETS:
    idx = idx_of_target[t]
    yt = Y[idx]; gt = G[idx]
    M = np.column_stack([gbm_oof[t][m] for m in ('lgb', 'xgb', 'cb')])
    Mte = np.column_stack([gbm_test[t][m] for m in ('lgb', 'xgb', 'cb')])
    oof = np.zeros(len(idx)); te_pred = np.zeros(len(Xte))
    for tr_idx, va_idx in GroupKFold(n_splits=GLOBAL_FOLDS).split(M, yt, gt):
        r = Ridge(alpha=1.0).fit(M[tr_idx], yt[tr_idx])
        oof[va_idx] = r.predict(M[va_idx])
        te_pred += r.predict(Mte) / GLOBAL_FOLDS
    stack_oof[t] = oof; stack_test[t] = te_pred

## 5. v16 Cross-Target Decoder — physics + learned arms (fold-safe)

In [ ]:
"""v16 Cross-Target Decoder — canonical, unit-testable source of truth.

The v16 Kaggle notebook embeds the decoder as one verbatim cell (the same
source-slice pattern as mt_gnn_v2.py -> CORE_A/CORE_B). Keeping it here as a
Python module lets the pure logic be unit-tested (tests/test_decoder_v16.py)
and lets vault/compare_v16.py reuse the physics math offline. Only train
labels are ever read (no test leakage). All folds use GroupKFold(n_splits=
GLOBAL_FOLDS) on `canon`, identical to P14.
"""
import numpy as np
import pandas as pd

GLOBAL_FOLDS = 5
SEED = 42

TARGETS_DEC = ["eea", "egb", "egc", "ei", "eps", "nc", "tg"]
TARGET_IDX_DEC = {t: i for i, t in enumerate(TARGETS_DEC)}

# Physics recipes: target -> (kind, srcs)
#   "subtract": out = src0 - src1         (egc = ei - eea)
#   "linear":   out = a * (feature(src)) + b, fitted from train pairs
#               egb = a * egc + b    | eps = a * nc^2 + b
PHYS_RECIPE = {
    "egc": ("subtract", ("ei", "eea")),
    "egb": ("linear", ("egc",)),
    "eps": ("linear", ("nc2",)),
}


def build_pivot_df(canon_arr, tgt_arr, val_arr):
    """Pivot table: index=canon, columns=TARGETS_DEC, values=target (NaN if absent)."""
    df = pd.DataFrame({"canon": canon_arr, "target_type": tgt_arr, "value": val_arr})
    piv = df.dropna(subset=["value"]).pivot_table(
        index="canon", columns="target_type", values="value", aggfunc="first")
    return piv.reindex(columns=TARGETS_DEC)


def sibling_feature(canon_list, pivot):
    """(n,7) float64 — for every row, the canon's 7 train-mediated sibling
    values in TARGETS_DEC column order (NaN where a target is absent)."""
    out = np.full((len(canon_list), 7), np.nan, dtype=np.float64)
    for i, c in enumerate(canon_list):
        if c in pivot.index:
            out[i] = pivot.loc[c].values
    return out


def _fit_linear(x, y):
    """Least-squares slope/intercept fit. Needs >= 3 points, else identity."""
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    n = len(x)
    if n < 3:
        return 1.0, 0.0
    A = np.vstack([x, np.ones(n)]).T
    slope, intercept = np.linalg.lstsq(A, y, rcond=None)[0]
    return float(slope), float(intercept)


def _aug_feature(sib):
    """Append derived feature columns to a (n,7) sibling matrix.
    Returns (aug, name->col) with 'nc2' = nc**2 appended at index 7."""
    nc2 = np.full(len(sib), np.nan, dtype=np.float64)
    ok = np.isfinite(sib[:, TARGET_IDX_DEC["nc"]])
    nc2[ok] = sib[ok, TARGET_IDX_DEC["nc"]] ** 2
    aug = np.column_stack([sib, nc2])
    names = {t: i for i, t in enumerate(TARGETS_DEC)}
    names["nc2"] = 7
    return aug, names


def physics_arm(sib_tr, sib_te, tr_tgt=None, group=None, global_folds=GLOBAL_FOLDS):
    """Return (phys_tr, phys_te) as (n_tr,7) and (n_te,7) float64 arrays in
    TARGETS_DEC column order. Missing/NaN stays NaN (caller falls back).

    Fold-safe: when `group` (array of per-row canon/labels) is given, the
    linear-recipe pair fits exclude the held-out fold's own pairs for the OOF
    arms (GroupKFold(global_folds) on group), and test uses a fit on all train
    pairs (train labels only). When group is None, a single all-train fit is
    used for both (unit-test convenience; still train-only).
    """
    n_tr, n_te = len(sib_tr), len(sib_te)
    out_tr = np.full((n_tr, 7), np.nan, dtype=np.float64)
    out_te = np.full((n_te, 7), np.nan, dtype=np.float64)

    aug_tr, names_tr = _aug_feature(sib_tr)
    aug_te, names_te = _aug_feature(sib_te)

    # per-fold membership of training rows (None when not fold-safe)
    if group is not None and global_folds > 1:
        from sklearn.model_selection import GroupKFold
        _cv = GroupKFold(n_splits=min(global_folds, len(np.unique(group))))
        fold_id = np.empty(n_tr, dtype=int)
        for _g, (_, vk) in enumerate(_cv.split(np.zeros(n_tr), np.zeros(n_tr), group)):
            fold_id[vk] = _g
    else:
        fold_id = np.zeros(n_tr, dtype=int)

    for tcol, (kind, srcs) in PHYS_RECIPE.items():
        ti = TARGET_IDX_DEC[tcol]
        if kind == "subtract":
            s0, s1 = (TARGET_IDX_DEC[s] for s in srcs)
            tr_ok = np.isfinite(sib_tr[:, s0]) & np.isfinite(sib_tr[:, s1])
            te_ok = np.isfinite(sib_te[:, s0]) & np.isfinite(sib_te[:, s1])
            out_tr[tr_ok, ti] = sib_tr[tr_ok, s0] - sib_tr[tr_ok, s1]
            out_te[te_ok, ti] = sib_te[te_ok, s0] - sib_te[te_ok, s1]
        else:  # linear
            fcol = names_tr[srcs[0]]
            # pairs where the source feature AND the destination value are known
            pair_ok_tr = np.isfinite(aug_tr[:, fcol]) & np.isfinite(sib_tr[:, ti])
            dst = sib_tr[pair_ok_tr, ti]
            srcv = aug_tr[pair_ok_tr, fcol]

            def _apply(a, b, aug):
                ok = np.isfinite(aug[:, fcol])
                vals = np.full(len(aug), np.nan, dtype=np.float64)
                vals[ok] = a * aug[ok, fcol] + b
                return vals

            if fold_id.max() == 0 or group is None:
                a, b = _fit_linear(srcv, dst)
                out_tr[:, ti] = _apply(a, b, aug_tr)
                out_te[:, ti] = _apply(a, b, aug_te)
            else:
                # fold-safe: per-fold coefficients on other-fold pairs
                a_all, b_all = _fit_linear(srcv, dst)
                out_te[:, ti] = _apply(a_all, b_all, aug_te)
                for k in range(fold_id.max() + 1):
                    keep = (fold_id != k) & pair_ok_tr
                    a_k, b_k = _fit_linear(aug_tr[:, fcol][keep],
                                           sib_tr[:, ti][keep])
                    m = fold_id == k
                    vals = np.full(n_tr, np.nan, dtype=np.float64)
                    vals[m] = a_k * aug_tr[m, fcol] + b_k
                    out_tr[:, ti] = np.where(m, vals, out_tr[:, ti])
    return out_tr, out_te


def learned_arm(canon_tr, tgt_tr, Y_tr, pivot, canon_te, global_folds=GLOBAL_FOLDS,
                seed=SEED, alpha=10.0, min_sibs=2):
    """Fold-safe learned cross-target arm. Returns (lo_tr, lo_te) as (n_tr,7)
    and (n_te,7) float64 arrays in TARGETS_DEC column order.

    For each target t: per-row features = the canon's sibling values for the
    other 6 targets (from the train-only pivot), standardized per fold. A
    per-target Ridge(alpha) is fit on the target-t rows of GroupKFold training
    folds and validated on the held-out canon-fold rows (a canon never crosses
    folds, so a held-out polymer's own target labels never enter its Ridge).
    Test inference averages the per-fold models on the full-train pivot.
    Rows whose canon has < `min_sibs` known siblings stay NaN (caller falls
    back to the target mean).
    """
    from sklearn.model_selection import GroupKFold
    from sklearn.linear_model import Ridge
    from sklearn.preprocessing import StandardScaler

    n_tr, n_te = len(canon_tr), len(canon_te)
    sib_tr = sibling_feature(canon_tr, pivot)
    sib_te = sibling_feature(canon_te, pivot)
    lo_tr = np.full((n_tr, 7), np.nan, dtype=np.float64)
    lo_te = np.full((n_te, 7), np.nan, dtype=np.float64)
    group_tr = np.asarray(canon_tr)

    for t in TARGETS_DEC:
        ti = TARGET_IDX_DEC[t]
        idx_t = np.where(tgt_tr == t)[0]
        if len(idx_t) < 1:
            continue
        keep_cols = [j for j in range(7) if j != ti]
        Ftr = sib_tr[:, keep_cols]
        Fte = sib_te[:, keep_cols]
        ok_tr = np.isfinite(Ftr).sum(axis=1) >= min_sibs
        ok_te = np.isfinite(Fte).sum(axis=1) >= min_sibs
        n_groups = len(np.unique(group_tr[idx_t]))
        if n_groups < 2:
            continue
        n_splits = max(1, min(global_folds, n_groups))
        cv = GroupKFold(n_splits=n_splits)
        te_acc = np.zeros(n_te)
        te_cnt = np.zeros(n_te)
        idx_t_arr = np.asarray(idx_t)
        for trk, vk in cv.split(idx_t_arr, Y_tr[idx_t], group_tr[idx_t]):
            # trk/vk are positional in idx_t; map to global rows
            g_trk = idx_t[trk]
            g_vk = idx_t[vk]
            fit_ok = g_trk[ok_tr[g_trk]]
            if len(fit_ok) < 1:
                continue
            # mean-impute the sibling columns on the training fold only
            # (fold-safe: val/test NaNs never influence the fit), drop
            # columns that are entirely NaN in this fold, then Ridge.
            Xf = Ftr[fit_ok].copy()
            fcol = np.isfinite(Xf).sum(axis=0) > 0
            Xf = Xf[:, fcol]
            if Xf.shape[1] < 1:
                continue
            colmean = np.nanmean(Xf, axis=0)
            colmean = np.where(np.isfinite(colmean), colmean, 0.0)
            Xf = np.where(np.isfinite(Xf), Xf, colmean)
            sc = StandardScaler().fit(Xf)
            m = Ridge(alpha=alpha).fit(sc.transform(Xf), Y_tr[fit_ok])
            vok = g_vk[ok_tr[g_vk]]
            if len(vok) > 0:
                Xv = Ftr[vok][:, fcol].copy()
                Xv = np.where(np.isfinite(Xv), Xv, colmean)
                lo_tr[vok, ti] = m.predict(sc.transform(Xv))
            if ok_te.any():
                Xt = Fte[ok_te][:, fcol].copy()
                Xt = np.where(np.isfinite(Xt), Xt, colmean)
                te_acc[ok_te] += m.predict(sc.transform(Xt))
                te_cnt[ok_te] += 1
        lo_te[:, ti] = np.where(te_cnt > 0, te_acc / np.maximum(te_cnt, 1), np.nan)
    return lo_tr, lo_te

In [ ]:
# decoder module re-declares GLOBAL_FOLDS=5/SEED=42 at import scope;
# restore the notebook run-mode values so SMOKE still uses 2 folds below.
GLOBAL_FOLDS = 2 if SMOKE else 5
SEED = 42

print("-- v16 decoder stage --", flush=True)
print("building pivot + sibling matrices from TRAIN labels only", flush=True)

tr_piv = build_pivot_df(trf["canon"].values, T.astype(str), Y)
sib_tr = sibling_feature(trf["canon"].values, tr_piv)      # (n_tr, 7)
sib_te = sibling_feature(tef["canon"].values, tr_piv)      # (n_te, 7)
phys_tr, phys_te = physics_arm(sib_tr, sib_te, group=G,
                               global_folds=GLOBAL_FOLDS)
lo_tr, lo_te = learned_arm(trf["canon"].values, T.astype(str), Y, tr_piv,
                           tef["canon"].values,
                           global_folds=GLOBAL_FOLDS, seed=SEED)
# physics always defined where requested; learned may be NaN for rare canons.
n_ok_p = int(np.isfinite(phys_te).sum()); n_ok_l = int(np.isfinite(lo_te).sum())
print("science_poly_te finite:", n_ok_p, "| learned_te finite:", n_ok_l, flush=True)
assert n_ok_p > 0, "physics arm produced all-NaN test predictions"


## 6. v16 blend — per-target Ridge [GBM, MT-GNN, physics, learned] OOF + submission

In [ ]:
ALPHA_GRID = [0.1, 0.5, 1.0, 2.5, 5.0, 10.0, 25.0]

oof_gbm_global = np.full(len(X), np.nan, dtype=np.float32)
oof_mt_global = np.full(len(X), np.nan, dtype=np.float32)
oof_phys_global = np.full(len(X), np.nan, dtype=np.float64)
oof_learn_global = np.full(len(X), np.nan, dtype=np.float64)
for t in TARGETS:
    idx = idx_of_target[t]
    ti = TARGET_IDX[t]
    oof_gbm_global[idx] = stack_oof[t]
    oof_mt_global[idx] = mt_oof[t]
    oof_phys_global[idx] = phys_tr[idx, ti]
    oof_learn_global[idx] = lo_tr[idx, ti]
assert not np.isnan(oof_gbm_global).any() and not np.isnan(oof_mt_global).any()

test_gbm_global = np.zeros(len(Xte), dtype=np.float32)
test_mt_global = np.zeros(len(Xte), dtype=np.float32)
test_phys_global = np.full(len(Xte), np.nan, dtype=np.float64)
test_learn_global = np.full(len(Xte), np.nan, dtype=np.float64)
for t in TARGETS:
    m_te = (tef["target_type"] == t).values
    ti = TARGET_IDX[t]
    test_gbm_global[m_te] = stack_test[t][m_te]
    test_mt_global[m_te] = mt_test[m_te]
    test_phys_global[m_te] = phys_te[:, ti][m_te]
    test_learn_global[m_te] = lo_te[:, ti][m_te]

def fillna_arm(v, fallback):
    """NaN in an arm row -> inert target-mean placeholder (neutral effect)."""
    v = np.asarray(v, dtype=np.float64)
    bad = ~np.isfinite(v)
    if bad.any():
        v = v.copy(); v[bad] = fallback
    return v

TMEAN = {t: float(np.nanmean(Y[idx_of_target[t]])) for t in TARGETS}

print("\n=== v16: tuning per-target Ridge alpha (4 arms) ===", flush=True)
rows = []
coefs = {t: [] for t in TARGETS}
best_a = {}
blends_tr = np.full(len(X), np.nan, dtype=np.float64)
blends_te = np.zeros(len(Xte), dtype=np.float64)
for t in TARGETS:
    idx = idx_of_target[t]
    ti = TARGET_IDX[t]
    idx_g = oof_gbm_global[idx]; idx_m = oof_mt_global[idx]
    Mt = np.column_stack([idx_g, idx_m,
                          fillna_arm(phys_tr[idx, ti], TMEAN[t]),
                          fillna_arm(lo_tr[idx, ti], TMEAN[t])])
    Mte = np.column_stack([test_gbm_global, test_mt_global,
                           fillna_arm(test_phys_global, TMEAN[t]),
                           fillna_arm(test_learn_global, TMEAN[t])])
    yt = Y[idx].astype(np.float64)
    cv = list(GroupKFold(n_splits=GLOBAL_FOLDS).split(Mt, yt, G[idx]))
    oof_r2 = {}
    for a in ALPHA_GRID:
        o = np.zeros(len(idx))
        for trk, vk in cv:
            o[vk] = Ridge(alpha=a).fit(Mt[trk], yt[trk]).predict(Mt[vk])
        oof_r2[a] = r2_score(yt, o)
    a_best = max(oof_r2, key=oof_r2.get)
    best_a[t] = a_best
    oof = np.zeros(len(idx)); te_pred = np.zeros(len(Xte))
    for trk, vk in cv:
        lr = Ridge(alpha=a_best); lr.fit(Mt[trk], yt[trk])
        oof[vk] = lr.predict(Mt[vk])
        te_pred += lr.predict(Mte) / GLOBAL_FOLDS
        coefs[t].append(lr.coef_.tolist())
    m_te = (tef["target_type"] == t).values
    blends_tr[idx] = oof
    blends_te[m_te] = te_pred[m_te]
    r_blend = r2_score(yt, oof); r_g = r2_score(yt, idx_g); r_m = r2_score(yt, idx_m)
    cb = np.mean(coefs[t], axis=0)
    rows.append(dict(target=t, alpha=float(a_best), blend=r_blend, GBM=r_g, GNN=r_m,
                     w_GBM=cb[0], w_GNN=cb[1], w_PH=cb[2], w_LEARN=cb[3]))
    print(f"  {t:<4} alpha={a_best:<6} blend={r_blend:.4f} GBM={r_g:.4f} GNN={r_m:.4f} "
          f"w_GMBM={cb[0]:.3f} w_GNN={cb[1]:.3f} w_PH={cb[2]:.3f} w_LEARN={cb[3]:.3f}", flush=True)

np.savez(os.path.join(OUT, "blend_oof_test16.npz"),
         oof_gbm=oof_gbm_global, oof_mt=oof_mt_global,
         oof_phys=oof_phys_global, oof_learn=oof_learn_global,
         test_gbm=test_gbm_global, test_mt=test_mt_global,
         test_phys=test_phys_global, test_learn=test_learn_global,
         blends_tr=blends_tr, blends_te=blends_te,
         y_all=Y.astype(np.float64), g_all=G.astype(str), t_all=T.astype(str),
         TMEAN_y=[TMEAN[t] for t in TARGETS])
print("wrote blend_oof_test16.npz", flush=True)

df = pd.DataFrame(rows).set_index("target")
print("\n=== v16 summary ===", flush=True)
print("  mean blend=%.4f | GBM=%.4f | GNN=%.4f | delta-4arm-vs-2arm %+.4f" % (
    df["blend"].mean(), df["GBM"].mean(), df["GNN"].mean(),
    df["blend"].mean() - df[["GBM", "GNN"]].mean(axis=1).mean()), flush=True)

sub = pd.DataFrame({"id": tef["id"].values, "target": blends_te})
sub_path = os.path.join(OUT, "submission_v16.csv")
sub.to_csv(sub_path, index=False)
print("\nwrote", sub_path, flush=True)
print("  rows", len(sub), "| NaN", sub["target"].isna().sum(), flush=True)
df.round(4).to_csv(os.path.join(OUT, "v16_blend_report.csv"), index=True)
print("wrote", os.path.join(OUT, "v16_blend_report.csv"), flush=True)
